# LAB 3 - Complex Pipelines (Splits and Merges)

In this Lab, we will cover splitting and merging pipelines.

A splitting pipeline allows for demultiplexing or separating a data feed into multiple downstream tables based on a filter query. For example an event feed that reads in all events from an source can be split into individual tables based on an event type parameter.

A merging pipeline allows for data that comes in from multiple sources to combined into a single table. This can be particulary useful in cases where your sources are separated by region and should be combined into a single table for easy analysis.

#Initialize the Notebook

Run the cell below to populate the properties from the overrides we set up in Lab 1.

If this command fails, rerun Step 3 from the first lab. 

In [0]:
import json, os

# load the overrides file into a variable
overrides_path = "../.databricks/bundle/dev/variable-overrides.json"
with open(overrides_path, "r") as f:
    variable_overrides = json.load(f)

print(json.dumps(variable_overrides, indent=2))

#LAB 3 - Task 1 Split Bronze into Multiple Silver

In this task we will split our silver tables based on region.


Run the cell below to configure the onboarding file.





In [0]:
import json

sdp_meta_schema = variable_overrides["sdp_meta_schema"]
uc_catalog_name = variable_overrides["uc_catalog_name"]
conf_volume = variable_overrides["conf_volume"]

template_path = "../conf/templates/onboarding_lab3_task1.template.json"
onboarding_path = "../conf/lab3/onboarding_lab3_task1.json"

with open(template_path, "r") as f:
    onboarding = json.load(f)

if not isinstance(onboarding, list):
    onboarding = [onboarding]

for entry in onboarding:
    entry["bronze_database_dev"] = sdp_meta_schema
    entry["silver_database_dev"] = sdp_meta_schema
    entry["bronze_catalog_dev"] = uc_catalog_name
    entry["silver_catalog_dev"] = uc_catalog_name
    entry["source_details"] = {
        "source_database": f"{uc_catalog_name}.{sdp_meta_schema}",
        "source_table": "host_status_updates_bronze"
    }

with open(onboarding_path, "w") as f:
    json.dump(onboarding, f, indent=2)

print(f"Updated onboarding_lab3_task1.json: bronze_database_dev and silver_database_dev set to '{sdp_meta_schema}'")
print(json.dumps(onboarding, indent=2))

### Examine the Onboarding File

If you examine the onboarding file that we created in Step 1, you might notice that it appears to be a partial pipeline. That is correct, you can create a single pipeline from multiple onboarding files. This pipeline will depend on the bronze table that we created in Lab 2. 

You specify which onboarding file to deploy when you run the Onboarding job. If you set a task parameter, `overwrite` to `False`, it will merge the changes in to the existing dataflowspecs table instead of recreating the table. This mechanism will allow you to define pipelines in multiple onboarding files when they become large and difficult to maintain as a single file.


### Deploy the Bundle ###

Deploy the Bundle as normal so that it will upload the onboarding files to the workspace location.

Once deployed we will run the onboarding job, but we will set the onboarding task to deploy the new onboarding file.

Open up the sdp-meta onboarding job from the bundle then click into the `Tasks` tab.

Click on the task named `onboard_dataflowspecs` and change the following parameters:

- **onboarding_file_path**: locates the onboarding file to load
- **onboard_layer**: possible values `bronze`/`silver`/`bronze_silver`
- **overwrite**: When `True` this will overwrite the dataflowspecs table. Set this to `False` to merge the new entries into the existing table.

| field | value |
|-------|-------|
|`onboarding_file_path`| `/Volumes/sdp_meta_lab/sdp_meta_<YOUR USERNAME>/sdp_meta/conf/lab3/onboarding_lab3_task1.json`|
|`onboard_layer`|`silver`|
|`overwrite`|`False`|


Once the Task parameters are updated, run the onboarding job to update the dataflowspecs table.

You should end up with one entry in the bronze_dataflowspec table and three entries in the silver_dataflowspec table.



In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_dataflowspec_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.bronze_dataflowspec');

silver_dataflowspec_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.silver_dataflowspec');

display(bronze_dataflowspec_df)

display(silver_dataflowspec_df)

assert bronze_dataflowspec_df.count() == 1
assert silver_dataflowspec_df.count() == 3


##Run the Pipeline

Your pipeline should now look like this:

![image_1790638054317.png](./img/image_1790638054317.png "lab3 task 1 - 3 split tables")

In [0]:

uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_silver');
silver_us_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_us_silver');
silver_global_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_global_silver');

display(silver_df)
display(silver_us_df)
display(silver_global_df)

# assert
assert silver_us_df.count() > 0
assert silver_global_df.count() > 0

print("Lab 3 Task 1 passed.")

Great job completing this task. I'm sure you may have noticed that this allows you to have silver tables with different rows, transforms, and SCD types.


#Lab 3 - Task 2 Merging Multiple Sources

For this task we will merge multiple sources into a single Bronze table.


The field on the onboarding document that is used for merging multiple flows are cdc_apply_changes_flows. Adding these fields will change the way that the pipeline is processed.

These fields are:
- bronze_cdc_apply_changes_flows for merging at the bronze layer
- silver_cdc_apply_changes_flows for applying the merge at the silver layer

In the documentation you may notice a reference to `append_flows`. According to the [Databricks documentation](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-apply-changes), `cdc_apply_changes_flows` completely replaces `append_flows`. Make sure that your pipelines do not specify both settings on the same target.

It is important that your sources have the same schema and CDC configuration. If you are dealing with hetergenous sources with schemas that may skew over time, you will need to handle this by having each source write to its own bronze table and then be converted to the target schema before they can merged at the silver layer.

We will set up the onboarding file for this job below.

Use the following settings for the onboarding job after the onboarding file has been updated and deployed:
| field | value |
|-------|-------|
|`onboarding_file_path`| `/Volumes/sdp_meta_lab/sdp_meta_egrun/sdp_meta/conf/lab3/onboarding_lab3_task2.json`|
|`onboard_layer`|`bronze`|
|`overwrite`|`False`|

In [0]:
import json

sdp_meta_schema = variable_overrides["sdp_meta_schema"]
uc_catalog_name = variable_overrides["uc_catalog_name"]
conf_volume = variable_overrides["conf_volume"]

template_path = "../conf/templates/onboarding_lab3_task2.template.json"
onboarding_path = "../conf/lab3/onboarding_lab3_task2.json"

with open(template_path, "r") as f:
    onboarding = json.load(f)

if not isinstance(onboarding, list):
    onboarding = [onboarding]

for entry in onboarding:
    entry["bronze_database_dev"] = sdp_meta_schema
    entry["bronze_catalog_dev"] = uc_catalog_name

    for flow in entry["bronze_cdc_apply_changes_flows"]["flows"]:
        region = flow["source_details"]["source_path_dev"].split("/")[-1]
        flow["source_details"] = {
            "source_path_dev": f"/Volumes/{uc_catalog_name}/{sdp_meta_schema}/sdp_meta/data/lab3/{region}"
        }

with open(onboarding_path, "w") as f:
    json.dump(onboarding, f, indent=2)

print(f"Updated onboarding_lab3_task2.json: bronze_database_dev and silver_database_dev set to '{sdp_meta_schema}'")
print(json.dumps(onboarding, indent=2))

The data for this task comes from three folders at /data/lab3. The onboarding file will load each of these folders using different steps in the pipeline into a single bronze table.

Deploy the Bundle as normal so that it will upload the onboarding files to the workspace location.

Once deployed we will run the onboarding job, but we will set the onboarding task to deploy the new onboarding file.

Open up the sdp-meta onboarding job from the bundle then click into the `Tasks` tab.

Use the following settings for the onboarding task
| field | value |
|-------|-------|
|`onboarding_file_path`| `/Volumes/sdp_meta_lab/sdp_meta_egrun/sdp_meta/conf/lab3/onboarding_lab3_task2.json`|
|`onboard_layer`|`bronze`|
|`overwrite`|`False`|


Once the Task parameters are updated, run the onboarding job to update the dataflowspecs table.

You should end up with one entry in the bronze_dataflowspec table and three entries in the silver_dataflowspec table.

##Run the Pipeline

Your pipeline should now look like this:

![image_1790640231194.png](./img/image_1790640231194.png "Lab 3 Task 2")

#Lab 3 - Task 2 Validation

In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

merged_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_merged_bronze');

display(merged_df)

# assert
regions = set(row['region'] for row in merged_df.select("region").distinct().collect())
expected_regions = {"us-east", "us-west", "eu-west", "ap-south"}
assert expected_regions.issubset(regions), f"Missing regions: {expected_regions - regions}. Found: {regions}"

print("Lab 3 Task 2 passed.")

## Lab 3 Bonus Task

Replace the source Bronze table of your Silver pipelines with this new merged that was created in Lab 3. Remove the Bronze table and pipeline that was configured in Lab 1.

# LAB 3 COMPLETED!

Congratulations you have completed this module.

Review the Bonus - Advanced Concepts module to learn about some of the additional features and capabilities of SDP-META.

### Features of SDP-Meta Covered in Labs 1–3

- **Lab 1: Environment Setup & Configuration**
  - Setting up variable overrides and workspace configuration
  - Configuring Unity Catalog (catalog, schema, and volumes)
  - Deploying bundles to the Databricks workspace

- **Lab 2: Basic Pipelines & Onboarding**
  - Creating and configuring onboarding files from templates
  - Onboarding dataflowspecs for bronze and silver layers
  - Running ingestion pipelines with bronze and silver tables
  - CDC (Change Data Capture) processing with SCD types
  - Using the `bronze_dataflowspec` and `silver_dataflowspec` tables

- **Lab 3: Complex Pipelines (Splits and Merges)**
  - **Splitting pipelines**: Demultiplexing a single source into multiple downstream silver tables based on filter queries (e.g., splitting by region)
  - **Merging pipelines**: Combining multiple sources into a single bronze table using `bronze_cdc_apply_changes_flows`
  - Incremental onboarding with `overwrite: False` to merge new entries into existing dataflowspec tables
  - Composing pipelines from multiple onboarding files
  - Configuring per-table transforms, SCD types, and filter queries at the silver layer

